# CityLearn 2022 — Transition Dataset Extraction

Collects `(s_t, a_t, s_{t+1})` tuples for all 17 buildings over 8760 timesteps.
Final dataset: **8760 × 17 = 148,920 transition records**, one per building per timestep.

Each row in the saved DataFrame has:
- `building_id` — which building (0–16)
- `timestep` — simulation hour (0–8759)
- `state` — observation vector at t
- `action` — action applied at t
- `next_state` — resulting observation at t+1

In [1]:
!pip install citylearn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 55.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.5/398.5 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. Thi

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "numpy==1.26.4",
    "pandas==2.2.2",
    "citylearn"
])

# Now restart the runtime
import os
os.kill(os.getpid(), 9)

In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

from citylearn.citylearn import CityLearnEnv

In [28]:
# ------------------------------------------------------------------
# 1. Init environment — multi-agent mode (central_agent=False)
#    This means observations and actions are returned as a LIST,
#    one entry per building. That's exactly what we want.
# ------------------------------------------------------------------
env = CityLearnEnv(
    schema='citylearn_challenge_2022_phase_all',
    central_agent=False   # <-- critical: gives per-building obs/actions
)

n_buildings = len(env.buildings)
print(f"Number of buildings: {n_buildings}")
print(f"Building names: {[b.name for b in env.buildings]}")
print(f"Obs space per building: {[obs.shape for obs in env.observation_space]}")
print(f"Action space per building: {[act.shape for act in env.action_space]}")

INFO:root:Go here /root/.cache/citylearn/v2.5.0/datasets/citylearn_challenge_2022_phase_all/schema.json 


Number of buildings: 17
Building names: ['Building_1', 'Building_2', 'Building_3', 'Building_4', 'Building_5', 'Building_6', 'Building_7', 'Building_8', 'Building_9', 'Building_10', 'Building_11', 'Building_12', 'Building_13', 'Building_14', 'Building_15', 'Building_16', 'Building_17']
Obs space per building: [(28,), (28,), (28,), (28,), (28,), (28,), (28,), (28,), (28,), (28,), (28,), (28,), (28,), (28,), (28,), (28,), (28,)]
Action space per building: [(1,), (1,), (1,), (1,), (1,), (1,), (1,), (1,), (1,), (1,), (1,), (1,), (1,), (1,), (1,), (1,), (1,)]


In [35]:
records = []
num_episodes_to_run = 20

for episode_num in range(num_episodes_to_run):
    states_raw = env.reset()
    states = states_raw[0]   # now a list of 17 lists

    # Simulate for up to 8760 timesteps within each episode
    for t in range(8760):

        actions = [space.sample() for space in env.action_space]

        step_output = env.step(actions)
        next_states = step_output[0][0] if isinstance(step_output[0], tuple) else step_output[0]
        rewards     = step_output[1]
        terminated  = step_output[2]

        for b_idx in range(n_buildings):
            records.append({
                'building_id':   b_idx,
                'building_name': env.buildings[b_idx].name,
                'episode_id':    episode_num, # Incrementing episode_id for each episode
                'timestep':      t,
                'state':         np.array(states[b_idx],      dtype=np.float32),
                'action':        np.array(actions[b_idx],     dtype=np.float32),
                'next_state':    np.array(next_states[b_idx], dtype=np.float32),
                'reward':        float(rewards[b_idx]),
            })

        states = next_states

        if terminated:
            print(f"Episode {episode_num} ended early at timestep {t}")
            break

print(f"\nTotal transition records collected: {len(records)}")
print(f"Expected (approx): {n_buildings} \u00d7 {num_episodes_to_run} \u00d7 8760 = {n_buildings * num_episodes_to_run * 8760}")

Episode 0 ended early at timestep 8758
Episode 1 ended early at timestep 8758
Episode 2 ended early at timestep 8758
Episode 3 ended early at timestep 8758
Episode 4 ended early at timestep 8758
Episode 5 ended early at timestep 8758
Episode 6 ended early at timestep 8758
Episode 7 ended early at timestep 8758
Episode 8 ended early at timestep 8758
Episode 9 ended early at timestep 8758
Episode 10 ended early at timestep 8758
Episode 11 ended early at timestep 8758
Episode 12 ended early at timestep 8758
Episode 13 ended early at timestep 8758
Episode 14 ended early at timestep 8758
Episode 15 ended early at timestep 8758
Episode 16 ended early at timestep 8758
Episode 17 ended early at timestep 8758
Episode 18 ended early at timestep 8758
Episode 19 ended early at timestep 8758

Total transition records collected: 2978060
Expected (approx): 17 × 20 × 8760 = 2978400


In [36]:
# ------------------------------------------------------------------
# 3. Build DataFrame and inspect
# ------------------------------------------------------------------
df = pd.DataFrame(records)
df = df[['state', 'action', 'next_state', 'episode_id']]

print(df.shape)
print(df.dtypes)

(2978060, 4)
state         object
action        object
next_state    object
episode_id     int64
dtype: object


In [47]:
# print first row of df
#print(df.iloc[0])
# include 'state' and 'action' columns
print(df.iloc[0, [0, 1, 2, 3]])

state         [7.0, 7.0, 24.0, 20.0, 18.3, 22.8, 20.0, 84.0,...
action                                             [0.20443559]
next_state    [8.0, 1.0, 1.0, 20.1, 19.4, 22.8, 19.4, 79.0, ...
episode_id                                                    0
Name: 0, dtype: object


In [48]:
# see distinct values of 'episode_id'
print(df['episode_id'].unique())

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


In [38]:
df.head(50)

,state,action,next_state,episode_id
0,"[7.0, 7.0, 24.0, 20.0, 18.3, 22.8, 20.0, 84.0,...",[0.20443559],"[8.0, 1.0, 1.0, 20.1, 19.4, 22.8, 19.4, 79.0, ...",0
1,"[7.0, 7.0, 24.0, 20.0, 18.3, 22.8, 20.0, 84.0,...",[-0.6726186],"[8.0, 1.0, 1.0, 20.1, 19.4, 22.8, 19.4, 79.0, ...",0
2,"[7.0, 7.0, 24.0, 20.0, 18.3, 22.8, 20.0, 84.0,...",[-0.71209234],"[8.0, 1.0, 1.0, 20.1, 19.4, 22.8, 19.4, 79.0, ...",0
3,"[7.0, 7.0, 24.0, 20.0, 18.3, 22.8, 20.0, 84.0,...",[0.8497852],"[8.0, 1.0, 1.0, 20.1, 19.4, 22.8, 19.4, 79.0, ...",0
4,"[7.0, 7.0, 24.0, 20.0, 18.3, 22.8, 20.0, 84.0,...",[-0.22190443],"[8.0, 1.0, 1.0, 20.1, 19.4, 22.8, 19.4, 79.0, ...",0
5,"[7.0, 7.0, 24.0, 20.0, 18.3, 22.8, 20.0, 84.0,...",[0.63458604],"[8.0, 1.0, 1.0, 20.1, 19.4, 22.8, 19.4, 79.0, ...",0
6,"[7.0, 7.0, 24.0, 20.0, 18.3, 22.8, 20.0, 84.0,...",[0.7826814],"[8.0, 1.0, 1.0, 20.1, 19.4, 22.8, 19.4, 79.0, ...",0
7,"[7.0, 7.0, 24.0, 20.0, 18.3, 22.8, 20.0, 84.0,...",[-0.23204814],"[8.0, 1.0, 1.0, 20.1, 19.4, 22.8, 19.4, 79.0, ...",0
8,"[7.0, 7.0, 24.0, 20.0, 18.3, 22.8, 20.0, 84.0,...",[0.9184502],"[8.0, 1.0, 1.0, 20.1, 19.4, 22.8, 19.4, 79.0, ...",0
9,"[7.0, 7.0, 24.0, 20.0, 18.3, 22.8, 20.0, 84.0,...",[0.46747062],"[8.0, 1.0, 1.0, 20.1, 19.4, 22.8, 19.4, 79.0, ...",0


In [39]:
# ------------------------------------------------------------------
# 4. Save — two options:
#    (a) Full flat DataFrame as pickle — fast for training
#    (b) Per-building .pkl files — clean for inspection
# ------------------------------------------------------------------

# Option A: single flat file — all 148920 rows
df.to_pickle('transitions_all_buildings.pkl')
print("Saved: transitions_all_buildings.pkl")

Saved: transitions_all_buildings.pkl


In [ ]:
# Option B: per-building files
import os
os.makedirs('rollouts', exist_ok=True)

for b_idx in range(n_buildings):
    b_df = df[df['building_id'] == b_idx].reset_index(drop=True)
    fname = f"rollouts/building_{b_idx:02d}.pkl"
    b_df.to_pickle(fname)

print(f"Saved {n_buildings} per-building rollout files to rollouts/")

In [40]:
print(f"Type of df['state'].iloc[0]: {type(df['state'].iloc[0])}")
print(f"Type of df['action'].iloc[0]: {type(df['action'].iloc[0])}")
print(f"Type of df['next_state'].iloc[0]: {type(df['next_state'].iloc[0])}")

Type of df['state'].iloc[0]: <class 'numpy.ndarray'>
Type of df['action'].iloc[0]: <class 'numpy.ndarray'>
Type of df['next_state'].iloc[0]: <class 'numpy.ndarray'>


The output confirms that the columns 'state', 'action', and 'next_state' indeed store `numpy.ndarray` objects, not strings. This ensures that the data is in the correct format for efficient training.